# CHARTA Step 34 — Fine-tune ClinicalBERT with LoRA on BC5CDR (CID Relation Extraction)

This notebook fine-tunes `emilyalsentzer/Bio_ClinicalBERT` for Chemical-Induced Disease (CID)
relation extraction using LoRA on the BC5CDR dataset. Designed for **Google Colab T4 GPU**.

**Workflow:**
1. Install dependencies
2. Clone CHARTA repo & set up path
3. Run fine-tuning
4. Download LoRA weights to local machine
5. (Optional) Evaluate NER F1

After downloading, place weights in `CHARTA/models/lora_weights/clinicalbert_rel/` locally.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title Step 1 — Install dependencies  {display-mode: "form"}
!pip install -q transformers datasets peft accelerate scispacy scikit-learn
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
print("✅ Dependencies installed")

  Preparing metadata (setup.py) ... done
✅ Dependencies installed


In [ ]:
#@title Step 2 — Load CHARTA from Google Drive & set PYTHONPATH {display-mode: "form"}

import os, sys
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Remove old extracted project
!rm -rf /content/CHARTA

# Copy ZIP from Drive
!cp /content/drive/MyDrive/CHARTA.zip /content/

# Extract ZIP quietly
!unzip -oq /content/CHARTA.zip -d /content/

# Set project root
CHARTA_ROOT = "/content/CHARTA"

# Verify project exists
assert os.path.isdir(CHARTA_ROOT), f"CHARTA not found at {CHARTA_ROOT}"

# Set PYTHONPATH
sys.path.insert(0, os.path.join(CHARTA_ROOT, "src"))

# Change working directory
os.chdir(CHARTA_ROOT)

print(f"✅ PYTHONPATH set, working dir: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ PYTHONPATH set, working dir: /content/CHARTA


In [ ]:
#@title Step 3 — Verify GPU  {display-mode: "form"}

import torch

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected — switch Colab runtime to T4 GPU")

PyTorch version : 2.10.0+cu128
GPU available   : True
GPU name        : Tesla T4
GPU memory      : 15.6 GB


In [ ]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

datasets: 2.19.1
huggingface_hub: 1.14.0


In [ ]:
#@title Step 4 — Fine-tune ClinicalBERT with LoRA {display-mode: "form"}
import sys

# ── clear cached modules ──────────────────────────────────────────────────
for key in list(sys.modules.keys()):
    if "layer2" in key:
        del sys.modules[key]

# ── imports ───────────────────────────────────────────────────────────────
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType

# ── config ────────────────────────────────────────────────────────────────
CLINICALBERT_MODEL = "emilyalsentzer/Bio_ClinicalBERT"

LORA_RANK = 8
LORA_ALPHA = 16
LORA_TARGET_MODULES = ["query", "value"]

OUTPUT_DIR = "models/lora_weights/clinicalbert_rel/"

# Faster training setup
NUM_EPOCHS = 3
MAX_LENGTH = 128

# ── load dataset ──────────────────────────────────────────────────────────
print("Loading BC5CDR dataset...")

ds = load_dataset(
    "bigbio/bc5cdr",
    "bc5cdr_bigbio_kb",
    trust_remote_code=True
)

# ── build examples ────────────────────────────────────────────────────────
def build_examples(split):

    examples = []

    for item in split:

        all_entities = item.get("entities", [])

        diseases = [
            e for e in all_entities
            if e["type"] == "Disease"
        ]

        chemicals = [
            e for e in all_entities
            if e["type"] == "Chemical"
        ]

        relations = {
            (r["arg1_id"], r["arg2_id"])
            for r in item.get("relations", [])
        }

        # Merge passage text
        passage_texts = []

        for p in item.get("passages", []):

            pt = p.get("text", "")

            if isinstance(pt, list):
                passage_texts.append(" ".join(pt))
            else:
                passage_texts.append(str(pt))

        full_text = " ".join(passage_texts).strip()

        # Create chemical-disease pairs
        for chem in chemicals:
            for dis in diseases:

                label = 1 if (
                    (chem["id"], dis["id"]) in relations or
                    (dis["id"], chem["id"]) in relations
                ) else 0

                chem_text = (
                    chem["text"][0]
                    if isinstance(chem["text"], list)
                    else chem["text"]
                )

                dis_text = (
                    dis["text"][0]
                    if isinstance(dis["text"], list)
                    else dis["text"]
                )

                input_text = (
                    f"[E1] {chem_text} [/E1] "
                    f"{full_text} "
                    f"[E2] {dis_text} [/E2]"
                )

                examples.append({
                    "text": input_text,
                    "label": label
                })

    return examples

# ── create datasets ───────────────────────────────────────────────────────
print("Building training examples...")
train_rows = build_examples(ds["train"])

print("Building validation examples...")
val_rows = build_examples(ds["validation"])

print(f"Original Train Size: {len(train_rows)}")
print(f"Original Val Size: {len(val_rows)}")

# ── debugging subset (faster training) ───────────────────────────────────
# Remove these later for full training
train_rows = train_rows[:2000]
val_rows = val_rows[:500]

print(f"Using Train Size: {len(train_rows)}")
print(f"Using Val Size: {len(val_rows)}")

assert len(train_rows) > 0, "Dataset is empty!"

# ── tokenizer ─────────────────────────────────────────────────────────────
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    CLINICALBERT_MODEL
)

# ── tokenization ──────────────────────────────────────────────────────────
def tokenize(batch):

    encoded = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    encoded["labels"] = batch["label"]

    return encoded

print("Tokenizing training dataset...")

train_ds = Dataset.from_list(train_rows).map(
    tokenize,
    batched=True
)

print("Tokenizing validation dataset...")

val_ds = Dataset.from_list(val_rows).map(
    tokenize,
    batched=True
)

print("Dataset columns:", train_ds.column_names)

# ── torch format ──────────────────────────────────────────────────────────
train_ds.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_ds.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# ── load model ────────────────────────────────────────────────────────────
print("Loading ClinicalBERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    CLINICALBERT_MODEL,
    num_labels=2
)

# ── LoRA configuration ────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_cfg)

print("\nTrainable Parameters:")
model.print_trainable_parameters()

# ── training arguments ────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-4,
    weight_decay=0.01,

    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    fp16=True,

    report_to="none"
)

# ── trainer ───────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

# ── training ──────────────────────────────────────────────────────────────
print("\nStarting Training...\n")

trainer.train()

# ── save model ────────────────────────────────────────────────────────────
print("\nSaving model...")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nTraining complete!")
print(f"Saved model to: {OUTPUT_DIR}")

Python path fixed!
Using src path: /content/src


2026-05-14 11:33:30,678 [INFO] layer2.trainer: Loading BC5CDR dataset ...
INFO:layer2.trainer:Loading BC5CDR dataset ...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for bigbio/bc5cdr contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bigbio/bc5cdr
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

trainable params: 296,450 || all params: 108,608,260 || trainable%: 0.2730


ValueError: Columns ['input_ids', 'label', 'attention_mask'] not in the dataset. Current columns in the dataset: []

In [ ]:
import sys
# Remove all cached layer2 modules
for key in list(sys.modules.keys()):
    if "layer2" in key:
        del sys.modules[key]

# ── imports ───────────────────────────────────────────────────────────────
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, TaskType

# ── config (inline, no imports from layer2.config) ────────────────────────
CLINICALBERT_MODEL    = "emilyalsentzer/Bio_ClinicalBERT"
LORA_RANK             = 8
LORA_ALPHA            = 16
LORA_TARGET_MODULES   = ["query", "value"]
OUTPUT_DIR            = "models/lora_weights/clinicalbert_rel/"
NUM_EPOCHS            = 10

# ── load dataset ──────────────────────────────────────────────────────────
ds = load_dataset("bigbio/bc5cdr", "bc5cdr_bigbio_kb", trust_remote_code=True)

# ── build examples ────────────────────────────────────────────────────────
def build_examples(split):
    examples = []
    for item in split:
        all_entities = item.get("entities", [])
        diseases  = [e for e in all_entities if e["type"] == "Disease"]
        chemicals = [e for e in all_entities if e["type"] == "Chemical"]
        rels = {(r["arg1_id"], r["arg2_id"]) for r in item.get("relations", [])}

        passage_texts = []
        for p in item.get("passages", []):
            pt = p.get("text", "")
            passage_texts.append(" ".join(pt) if isinstance(pt, list) else str(pt))
        full_text = " ".join(passage_texts).strip()

        for c in chemicals:
            for d in diseases:
                label = 1 if (c["id"], d["id"]) in rels or \
                             (d["id"], c["id"]) in rels else 0
                c_text = c["text"][0] if isinstance(c["text"], list) else c["text"]
                d_text = d["text"][0] if isinstance(d["text"], list) else d["text"]
                inp = f"[E1] {c_text} [/E1] {full_text} [E2] {d_text} [/E2]"
                examples.append({"text": inp, "label": label})
    return examples

train_rows = build_examples(ds["train"])
val_rows   = build_examples(ds["validation"])
print(f"Train: {len(train_rows)} | Val: {len(val_rows)}")
assert len(train_rows) > 0, "Still empty — paste output here"

# ── tokenize ──────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(CLINICALBERT_MODEL)

def tokenize(batch):
    enc = tokenizer(batch["text"], truncation=True,
                    padding="max_length", max_length=256)
    enc["label"] = batch["label"]
    return enc

train_ds = Dataset.from_list(train_rows).map(tokenize, batched=True)
val_ds   = Dataset.from_list(val_rows).map(tokenize, batched=True)
print(f"Columns: {train_ds.column_names}")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch",   columns=["input_ids", "attention_mask", "label"])

# ── model + LoRA ──────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    CLINICALBERT_MODEL, num_labels=2)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES, lora_dropout=0.1, bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── train ─────────────────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    learning_rate=2e-4, weight_decay=0.01, fp16=True,
    logging_steps=50, report_to="none",
)
trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Done. Saved to {OUTPUT_DIR}")

Train: 44209 | Val: 47502


Map:   0%|          | 0/44209 [00:00<?, ? examples/s]

Map:   0%|          | 0/47502 [00:00<?, ? examples/s]

Columns: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

trainable params: 296,450 || all params: 108,608,260 || trainable%: 0.2730


Epoch,Training Loss,Validation Loss
1,0.458341,0.669073
2,0.384875,0.679434
3,0.335281,0.777790
4,0.314058,0.780164
5,0.330402,0.825401
6,0.312183,0.858381
7,0.296115,0.905343
8,0.298518,0.900796


In [ ]:
#@title Step 5 — Verify saved LoRA weights  {display-mode: "form"}
import os

weights_dir = "models/lora_weights/clinicalbert_rel/"
expected_files = ["adapter_config.json", "adapter_model.safetensors"]
for f in expected_files:
    path = os.path.join(weights_dir, f)
    assert os.path.isfile(path), f"Missing: {path}"
    size_kb = os.path.getsize(path) / 1024
    print(f"  ✅ {f} ({size_kb:.1f} KB)")

# Also check tokenizer files
tok_files = ["tokenizer.json", "tokenizer_config.json", "vocab.txt"]
for f in tok_files:
    path = os.path.join(weights_dir, f)
    if os.path.isfile(path):
        print(f"  ✅ {f} ({os.path.getsize(path)/1024:.1f} KB)")

print("\n✅ All LoRA weight files saved successfully")

In [ ]:
#@title Step 6 — Test Relation Extraction Model {display-mode: "form"}

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from peft import PeftModel
import torch

# ── base model ────────────────────────────────────────────────────────────
CLINICALBERT_MODEL = "emilyalsentzer/Bio_ClinicalBERT"

# ── load tokenizer ────────────────────────────────────────────────────────
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "models/lora_weights/clinicalbert_rel/"
)

# ── load base model ───────────────────────────────────────────────────────
print("Loading base ClinicalBERT model...")

base_model = AutoModelForSequenceClassification.from_pretrained(
    CLINICALBERT_MODEL,
    num_labels=2
)

# ── load LoRA adapter ─────────────────────────────────────────────────────
print("Loading LoRA weights...")

model = PeftModel.from_pretrained(
    base_model,
    "models/lora_weights/clinicalbert_rel/"
)

# ── evaluation mode ───────────────────────────────────────────────────────
model.eval()

# ── sample biomedical relation test ───────────────────────────────────────
test_input = (
    "[E1] aspirin [/E1] "
    "The patient was treated with aspirin for "
    "[E2] myocardial infarction [/E2]"
)

print("\nTest Input:")
print(test_input)

# ── tokenize input ────────────────────────────────────────────────────────
inputs = tokenizer(
    test_input,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

# ── inference ─────────────────────────────────────────────────────────────
with torch.no_grad():
    outputs = model(**inputs)

# ── prediction ────────────────────────────────────────────────────────────
prediction = torch.argmax(outputs.logits, dim=1).item()

print("\nPrediction:", prediction)

if prediction == 1:
    print("✅ Relation detected")
else:
    print("❌ No relation detected")

In [ ]:
#@title Step 7 — (Optional) Evaluate NER F1  {display-mode: "form"}
from layer2.evaluator import evaluate_ner

# This evaluates scispaCy NER on NCBI Disease test split
# Expected: micro-F1 > 0.75
results = evaluate_ner()
print(f"NER F1: {results['ner_f1']:.4f}")

## Step 8 — Download LoRA weights to your local machine

Run the cell below to download the LoRA weights as a zip file. Then extract them into
`CHARTA/models/lora_weights/clinicalbert_rel/` on your local machine.

**Local folder structure after extraction:**
```
CHARTA/
├── models/
│   └── lora_weights/
│       └── clinicalbert_rel/
│           ├── adapter_config.json
│           ├── adapter_model.safetensors
│           ├── tokenizer.json
│           ├── tokenizer_config.json
│           └── vocab.txt
│           └── special_tokens_map.json
```

In [ ]:
#@title Step 8 — Download LoRA weights as zip  {display-mode: "form"}
import shutil
from google.colab import files

zip_path = shutil.make_archive("clinicalbert_rel_lora", "zip",
                              root_dir="models/lora_weights",
                              base_dir="clinicalbert_rel")
print(f"Zip created: {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)")
files.download(zip_path)
print("✅ Download started — extract into CHARTA/models/lora_weights/clinicalbert_rel/ locally")

## Post-Download: Local Setup Instructions

After downloading and extracting the zip into `CHARTA/models/lora_weights/clinicalbert_rel/`:

1. **Verify locally:**
   ```bash
   cd CHARTA/src
   python -c "from layer2.relation_extractor import load_relation_model; t, m = load_relation_model(); print('LoRA model loaded OK')"
   ```

2. **Run Layer 2 pipeline with relations:**
   ```bash
   python run_layer2.py --mode run
   ```

3. **The relation extractor will now return CID relations instead of empty lists.**